In [1]:
import torch
import torch.nn.functional as F

torch.manual_seed(0)

vocab_size = 5
target = torch.tensor([2])  # correct token is index 2

examples = {
    "No clue": torch.zeros(1, vocab_size),
    "Leans correct": torch.tensor([[0., 0., 3., 0., 0.]]),
    "Confidently wrong": torch.tensor([[5., 0., 0., 0., 0.]])
}

for name, logits in examples.items():
    probs = F.softmax(logits, dim=1)
    true_prob = probs[0, target.item()]
    loss = F.cross_entropy(logits, target)

    print(f"\n{name}")
    print("-" * len(name))
    print("Logits       :", logits.tolist()[0])
    print("Probabilities:", [round(p, 4) for p in probs[0].tolist()])
    print("P(correct)   :", round(true_prob.item(), 4))
    print("CrossEntropy = -log(P(correct))")
    print(f"             = -log({true_prob:.4f})")
    print(f"             = {loss.item():.4f}")


No clue
-------
Logits       : [0.0, 0.0, 0.0, 0.0, 0.0]
Probabilities: [0.2, 0.2, 0.2, 0.2, 0.2]
P(correct)   : 0.2
CrossEntropy = -log(P(correct))
             = -log(0.2000)
             = 1.6094

Leans correct
-------------
Logits       : [0.0, 0.0, 3.0, 0.0, 0.0]
Probabilities: [0.0415, 0.0415, 0.8339, 0.0415, 0.0415]
P(correct)   : 0.8339
CrossEntropy = -log(P(correct))
             = -log(0.8339)
             = 0.1816

Confidently wrong
-----------------
Logits       : [5.0, 0.0, 0.0, 0.0, 0.0]
Probabilities: [0.9738, 0.0066, 0.0066, 0.0066, 0.0066]
P(correct)   : 0.0066
CrossEntropy = -log(P(correct))
             = -log(0.0066)
             = 5.0266


In [3]:
import os, urllib.request, torch

if not os.path.exists("shakespeare.txt"):
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    urllib.request.urlretrieve(url, "shakespeare.txt")

text = open("shakespeare.txt").read()
print("corpus length:", len(text), "characters\n")
print(text[:180])

chars = sorted(set(text))
vocab_size = len(chars)
print("\nvocabulary:", vocab_size, "unique characters")
print("".join(chars))

corpus length: 1115394 characters

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First

vocabulary: 65 unique characters

 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz


In [4]:
stoi = {c: i for i, c in enumerate(chars)}      # char  -> id
itos = {i: c for c, i in stoi.items()}          # id    -> char
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: "".join(itos[i] for i in ids)

print("encode('Hello'):", encode("Hello"))
print("decode back    :", decode(encode("Hello")))

# encode the whole corpus, then hold out the last 10% for validation
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]
print("\ndata tensor:", tuple(data.shape), "| train:", len(train_data), "| val:", len(val_data))

encode('Hello'): [20, 43, 50, 50, 53]
decode back    : Hello

data tensor: (1115394,) | train: 1003854 | val: 111540


In [5]:
import torch
import torch.nn as nn
torch.manual_seed(1337)

text = open("shakespeare.txt").read()        # created in Lesson 2
chars = sorted(set(text)); vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
data = torch.tensor([stoi[c] for c in text], dtype=torch.long)

block_size = 8      # how many tokens of context the model sees at once
batch_size = 4      # sequences per batch
n_embd = 32         # size of each embedding vector

def get_batch(src, block, bs):
    ix = torch.randint(len(src) - block, (bs,))
    x = torch.stack([src[i:i + block] for i in ix])          # (bs, block)  inputs
    y = torch.stack([src[i + 1:i + block + 1] for i in ix])  # (bs, block)  targets = inputs shifted by 1
    return x, y

xb, yb = get_batch(data, block_size, batch_size)
print("input  batch x:", tuple(xb.shape), "(batch, time)")
print("target batch y:", tuple(yb.shape), "-> y is x shifted one step (the 'next token' at each position)")

input  batch x: (4, 8) (batch, time)
target batch y: (4, 8) -> y is x shifted one step (the 'next token' at each position)


In [6]:
token_embedding    = nn.Embedding(vocab_size, n_embd)   # what the token is
position_embedding = nn.Embedding(block_size, n_embd)   # where it sits

tok = token_embedding(xb)                                # (B, T, C)
pos = position_embedding(torch.arange(block_size))       # (T, C)
x = tok + pos                                            # broadcast -> (B, T, C)

print("token embeddings    :", tuple(tok.shape))
print("position embeddings :", tuple(pos.shape))
print("model input (tok+pos):", tuple(x.shape), " <- one 32-d vector per token, position baked in")

token embeddings    : (4, 8, 32)
position embeddings : (8, 32)
model input (tok+pos): (4, 8, 32)  <- one 32-d vector per token, position baked in


In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(1337)

# --- rebuild the embedded batch from Lessons 2-3 ---
text = open("shakespeare.txt").read()
chars = sorted(set(text)); vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
data = torch.tensor([stoi[c] for c in text], dtype=torch.long)
block_size, batch_size, n_embd = 8, 4, 32

def get_batch(src, block, bs):
    ix = torch.randint(len(src) - block, (bs,))
    return (torch.stack([src[i:i + block] for i in ix]),
            torch.stack([src[i + 1:i + block + 1] for i in ix]))

xb, yb = get_batch(data, block_size, batch_size)
x = nn.Embedding(vocab_size, n_embd)(xb) + nn.Embedding(block_size, n_embd)(torch.arange(block_size))
B, T, C = x.shape

# --- one attention head, from scratch ---
head_size = 16
key   = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

k, q, v = key(x), query(x), value(x)              # each (B, T, head_size)
wei = q @ k.transpose(-2, -1) * head_size ** -0.5  # (B, T, T) scaled scores
wei = F.softmax(wei, dim=-1)                        # each row sums to 1
out = wei @ v                                       # (B, T, head_size)

print("q, k, v          :", tuple(q.shape))
print("attention weights:", tuple(wei.shape), "(query token x key token)")
print("every row sums to 1:", bool(torch.allclose(wei.sum(-1), torch.ones(B, T))))
print("head output      :", tuple(out.shape))
print("\nattention matrix for one sequence (row i = how token i spreads its attention):")
print(wei[0].round(decimals=2))

q, k, v          : (4, 8, 16)
attention weights: (4, 8, 8) (query token x key token)
every row sums to 1: True
head output      : (4, 8, 16)

attention matrix for one sequence (row i = how token i spreads its attention):
tensor([[0.1900, 0.1200, 0.0800, 0.2300, 0.0700, 0.1400, 0.0700, 0.1100],
        [0.1100, 0.2100, 0.0800, 0.0300, 0.2300, 0.2000, 0.0600, 0.1000],
        [0.1300, 0.1100, 0.0100, 0.0700, 0.2500, 0.0900, 0.2000, 0.1400],
        [0.1000, 0.1000, 0.1500, 0.2300, 0.1400, 0.1000, 0.1100, 0.0700],
        [0.1500, 0.1100, 0.0700, 0.0700, 0.1300, 0.2700, 0.0400, 0.1500],
        [0.1300, 0.0900, 0.2200, 0.0800, 0.1400, 0.1300, 0.1100, 0.1000],
        [0.1100, 0.1100, 0.2400, 0.0800, 0.1000, 0.0600, 0.1900, 0.1100],
        [0.0800, 0.2600, 0.0800, 0.0900, 0.2300, 0.0900, 0.1000, 0.0600]],
       grad_fn=<RoundBackward1>)


In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(1337)

# --- rebuild the embedded batch from Lessons 2-3 ---
text = open("shakespeare.txt").read()
chars = sorted(set(text)); vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
data = torch.tensor([stoi[c] for c in text], dtype=torch.long)
block_size, batch_size, n_embd = 8, 4, 32

def get_batch(src, block, bs):
    ix = torch.randint(len(src) - block, (bs,))
    return (torch.stack([src[i:i + block] for i in ix]),
            torch.stack([src[i + 1:i + block + 1] for i in ix]))

xb, yb = get_batch(data, block_size, batch_size)
x = nn.Embedding(vocab_size, n_embd)(xb) + nn.Embedding(block_size, n_embd)(torch.arange(block_size))
B, T, C = x.shape

head_size = 16
key, query, value = (nn.Linear(C, head_size, bias=False) for _ in range(3))
k, q, v = key(x), query(x), value(x)
wei = q @ k.transpose(-2, -1) * head_size ** -0.5

print("BEFORE masking — token attends everywhere (note non-zero above the diagonal):")
print(F.softmax(wei, dim=-1)[0].round(decimals=2))

tril = torch.tril(torch.ones(T, T))                 # 1s on/below the diagonal
wei = wei.masked_fill(tril == 0, float("-inf"))     # future -> -inf
wei = F.softmax(wei, dim=-1)

print("\nAFTER causal mask — strictly lower-triangular; each token only looks back:")
print(wei[0].round(decimals=2))
out = wei @ v
print("\nrow 0 attends only to token 0:", wei[0, 0].round(decimals=2).tolist())

BEFORE masking — token attends everywhere (note non-zero above the diagonal):
tensor([[0.1900, 0.1200, 0.0800, 0.2300, 0.0700, 0.1400, 0.0700, 0.1100],
        [0.1100, 0.2100, 0.0800, 0.0300, 0.2300, 0.2000, 0.0600, 0.1000],
        [0.1300, 0.1100, 0.0100, 0.0700, 0.2500, 0.0900, 0.2000, 0.1400],
        [0.1000, 0.1000, 0.1500, 0.2300, 0.1400, 0.1000, 0.1100, 0.0700],
        [0.1500, 0.1100, 0.0700, 0.0700, 0.1300, 0.2700, 0.0400, 0.1500],
        [0.1300, 0.0900, 0.2200, 0.0800, 0.1400, 0.1300, 0.1100, 0.1000],
        [0.1100, 0.1100, 0.2400, 0.0800, 0.1000, 0.0600, 0.1900, 0.1100],
        [0.0800, 0.2600, 0.0800, 0.0900, 0.2300, 0.0900, 0.1000, 0.0600]],
       grad_fn=<RoundBackward1>)

AFTER causal mask — strictly lower-triangular; each token only looks back:
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3500, 0.6500, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5200, 0.4400, 0.0400, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000]

In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(1337)

# --- rebuild the embedded batch from Lessons 2-3 ---
text = open("shakespeare.txt").read()
chars = sorted(set(text)); vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
data = torch.tensor([stoi[c] for c in text], dtype=torch.long)
block_size, batch_size, n_embd = 8, 4, 32

def get_batch(src, block, bs):
    ix = torch.randint(len(src) - block, (bs,))
    return (torch.stack([src[i:i + block] for i in ix]),
            torch.stack([src[i + 1:i + block + 1] for i in ix]))

xb, yb = get_batch(data, block_size, batch_size)
x = nn.Embedding(vocab_size, n_embd)(xb) + nn.Embedding(block_size, n_embd)(torch.arange(block_size))
B, T, C = x.shape

class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
    def forward(self, x):
        B, T, C = x.shape
        k, q, v = self.key(x), self.query(x), self.value(x)
        wei = q @ k.transpose(-2, -1) * k.shape[-1] ** -0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wei = F.softmax(wei, dim=-1)
        return wei @ v

class MultiHeadAttention(nn.Module):
    def __init__(self, n_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(n_heads)])
        self.proj  = nn.Linear(n_heads * head_size, n_embd)
    def forward(self, x):
        return self.proj(torch.cat([h(x) for h in self.heads], dim=-1))

class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(n_embd, 4 * n_embd), nn.ReLU(), nn.Linear(4 * n_embd, n_embd))
    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd, n_heads):
        super().__init__()
        self.sa  = MultiHeadAttention(n_heads, n_embd // n_heads)
        self.ff  = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)
    def forward(self, x):
        x = x + self.sa(self.ln1(x))    # residual around pre-normed attention
        x = x + self.ff(self.ln2(x))    # residual around pre-normed feed-forward
        return x

block = Block(n_embd, n_heads=4)
out = block(x)
print("block input :", tuple(x.shape))
print("block output:", tuple(out.shape), " <- same shape in and out, so blocks stack")
print("parameters in one block:", sum(p.numel() for p in block.parameters()))

block input : (4, 8, 32)
block output: (4, 8, 32)  <- same shape in and out, so blocks stack
parameters in one block: 12608


In [10]:
import torch, torch.nn as nn, torch.nn.functional as F
torch.manual_seed(1337)

text = open("shakespeare.txt").read()
chars = sorted(set(text)); vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}; itos = {i: c for c, i in stoi.items()}
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: "".join(itos[i] for i in ids)
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data)); train_data, val_data = data[:n], data[n:]

# --- model config (small, so it trains on a CPU in a couple of minutes) ---
block_size = 32     # context length
n_embd     = 64     # embedding size
n_heads    = 4      # attention heads
n_layers   = 3      # transformer blocks
batch_size = 16

class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
    def forward(self, x):
        B, T, C = x.shape
        k, q, v = self.key(x), self.query(x), self.value(x)
        wei = q @ k.transpose(-2, -1) * k.shape[-1] ** -0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wei = F.softmax(wei, dim=-1)
        return wei @ v

class MultiHeadAttention(nn.Module):
    def __init__(self, n_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(n_heads)])
        self.proj  = nn.Linear(n_heads * head_size, n_embd)
    def forward(self, x):
        return self.proj(torch.cat([h(x) for h in self.heads], dim=-1))

class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(n_embd, 4 * n_embd), nn.ReLU(), nn.Linear(4 * n_embd, n_embd))
    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd, n_heads):
        super().__init__()
        self.sa  = MultiHeadAttention(n_heads, n_embd // n_heads)
        self.ff  = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd); self.ln2 = nn.LayerNorm(n_embd)
    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

class GPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding    = nn.Embedding(vocab_size, n_embd)
        self.position_embedding = nn.Embedding(block_size, n_embd)
        self.blocks  = nn.Sequential(*[Block(n_embd, n_heads) for _ in range(n_layers)])
        self.ln_f    = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)
    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.token_embedding(idx) + self.position_embedding(torch.arange(T))
        x = self.ln_f(self.blocks(x))
        logits = self.lm_head(x)                       # (B, T, vocab)
        if targets is None:
            return logits, None
        loss = F.cross_entropy(logits.view(B * T, -1), targets.view(B * T))
        return logits, loss
    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -block_size:])
            probs = F.softmax(logits[:, -1, :], dim=-1)
            idx = torch.cat([idx, torch.multinomial(probs, 1)], dim=1)
        return idx

model = GPT()
print("total parameters:", sum(p.numel() for p in model.parameters()))

# one forward pass before any training — loss should sit near log(vocab_size)
xb = torch.stack([train_data[i:i + block_size] for i in torch.randint(len(train_data) - block_size, (batch_size,))])
yb = torch.stack([train_data[i + 1:i + block_size + 1] for i in torch.randint(len(train_data) - block_size, (batch_size,))])
_, loss = model(xb, yb)
import math
print(f"untrained loss: {loss.item():.3f}   (random guessing = log({vocab_size}) = {math.log(vocab_size):.3f})")

total parameters: 159937
untrained loss: 4.450   (random guessing = log(65) = 4.174)


In [11]:
def get_batch(split):
    src = train_data if split == "train" else val_data
    ix = torch.randint(len(src) - block_size, (batch_size,))
    x = torch.stack([src[i:i + block_size] for i in ix])
    y = torch.stack([src[i + 1:i + block_size + 1] for i in ix])
    return x, y

@torch.no_grad()
def estimate_loss():
    model.eval(); out = {}
    for split in ["train", "val"]:
        losses = torch.zeros(10)
        for k in range(10):
            _, l = model(*get_batch(split)); losses[k] = l.item()
        out[split] = losses.mean().item()
    model.train(); return out

opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
max_iters, eval_interval = 400, 100
for it in range(max_iters + 1):
    if it % eval_interval == 0:
        l = estimate_loss(); print(f"step {it:4d}:  train loss {l['train']:.3f}   val loss {l['val']:.3f}")
    _, loss = model(*get_batch("train"))
    opt.zero_grad(set_to_none=True); loss.backward(); opt.step()

torch.save(model.state_dict(), "gpt.pt")     # save for Lessons 8-9
print("\nsaved gpt.pt")

step    0:  train loss 4.409   val loss 4.409
step  100:  train loss 2.518   val loss 2.517
step  200:  train loss 2.366   val loss 2.366
step  300:  train loss 2.274   val loss 2.289
step  400:  train loss 2.212   val loss 2.180

saved gpt.pt


In [12]:
context = torch.zeros((1, 1), dtype=torch.long)     # a single start token
print(decode(model.generate(context, max_new_tokens=400)[0].tolist()))


A knes, en myours Warm-lior sood'ss
qomsurmame whows, af my bumessir man and you dence;
lane aree amerp sould bins alt o sis morcen?
Whour Isdive ent soverectere not cet mad an
Wutt ous be the to it theraurr, maged:
Tise, mmes lord cose diviongs lond!
Hed oulto to wor Ind mutwithe of alge.
Theer tom faigesin onse hy enge to oldef,
Lewss sros of cik boirlood tein ansoved, taves he basettes Ghou thi


In [13]:
import torch, torch.nn as nn, torch.nn.functional as F
torch.manual_seed(1337)

text = open("shakespeare.txt").read()
chars = sorted(set(text)); vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}; itos = {i: c for c, i in stoi.items()}
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: "".join(itos[i] for i in ids)
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data)); train_data, val_data = data[:n], data[n:]

# --- model config (small, so it trains on a CPU in a couple of minutes) ---
block_size = 32     # context length
n_embd     = 64     # embedding size
n_heads    = 4      # attention heads
n_layers   = 3      # transformer blocks
batch_size = 16

class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
    def forward(self, x):
        B, T, C = x.shape
        k, q, v = self.key(x), self.query(x), self.value(x)
        wei = q @ k.transpose(-2, -1) * k.shape[-1] ** -0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wei = F.softmax(wei, dim=-1)
        return wei @ v

class MultiHeadAttention(nn.Module):
    def __init__(self, n_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(n_heads)])
        self.proj  = nn.Linear(n_heads * head_size, n_embd)
    def forward(self, x):
        return self.proj(torch.cat([h(x) for h in self.heads], dim=-1))

class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(n_embd, 4 * n_embd), nn.ReLU(), nn.Linear(4 * n_embd, n_embd))
    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd, n_heads):
        super().__init__()
        self.sa  = MultiHeadAttention(n_heads, n_embd // n_heads)
        self.ff  = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd); self.ln2 = nn.LayerNorm(n_embd)
    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

class GPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding    = nn.Embedding(vocab_size, n_embd)
        self.position_embedding = nn.Embedding(block_size, n_embd)
        self.blocks  = nn.Sequential(*[Block(n_embd, n_heads) for _ in range(n_layers)])
        self.ln_f    = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)
    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.token_embedding(idx) + self.position_embedding(torch.arange(T))
        x = self.ln_f(self.blocks(x))
        logits = self.lm_head(x)                       # (B, T, vocab)
        if targets is None:
            return logits, None
        loss = F.cross_entropy(logits.view(B * T, -1), targets.view(B * T))
        return logits, loss
    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -block_size:])
            probs = F.softmax(logits[:, -1, :], dim=-1)
            idx = torch.cat([idx, torch.multinomial(probs, 1)], dim=1)
        return idx

model = GPT(); model.load_state_dict(torch.load("gpt.pt")); model.eval()

@torch.no_grad()
def sample(max_new_tokens=300, temperature=1.0, top_k=None, top_p=None, seed=0):
    torch.manual_seed(seed)
    idx = torch.zeros((1, 1), dtype=torch.long)
    for _ in range(max_new_tokens):
        logits, _ = model(idx[:, -block_size:])
        logits = logits[:, -1, :] / temperature
        if top_k is not None:
            kth = torch.topk(logits, top_k)[0][:, [-1]]
            logits = logits.masked_fill(logits < kth, float("-inf"))
        if top_p is not None:
            s_logits, s_idx = torch.sort(logits, descending=True)
            cum = F.softmax(s_logits, dim=-1).cumsum(-1)
            drop = cum > top_p; drop[:, 1:] = drop[:, :-1].clone(); drop[:, 0] = False
            s_logits = s_logits.masked_fill(drop, float("-inf"))
            logits = torch.empty_like(logits).scatter_(-1, s_idx, s_logits)
        probs = F.softmax(logits, dim=-1)
        idx = torch.cat([idx, torch.multinomial(probs, 1)], dim=1)
    return decode(idx[0].tolist())

print("=== temperature 0.5 (conservative) ===\n", sample(temperature=0.5))

=== temperature 0.5 (conservative) ===
 
thy thou she wisell it of the hat youd thelst tou courese hit therere he he his bere offor on is seant of tor it bee and this could, not my my tood the sto for mat ises the cout cond soud the couse of to prate his mely my hond sour touse meld for my ind the me he ton sit in be then astices sould my 


In [14]:
print("=== temperature 1.4 (wild) ===\n", sample(temperature=1.4))
print("\n=== top-k = 10 ===\n", sample(top_k=10))
print("\n=== top-p = 0.9 (nucleus) ===\n", sample(top_p=0.9))

=== temperature 1.4 (wild) ===
 
tly, mhis khexpselter?ouce meaman youd Igelst, grecooudsy hitnty
here hath Dom seeiell! sanser, Cit.
---Ky wreit Myemaccesen;
JUf Ra no- my fittoor epe you fori'd Iioedvott,
Prescon' souz thul ne.Sevack, pradithi
-Olly mooven, siay, int ouqld fleomewigl' i'd tout tonc.
Poin,bo youn qon gor evexmysha

=== top-k = 10 ===
 
Thy, mhis thereselt ild canderave woud woulste trecoures,
O thaverere hath thic be hase sand him;s watsoos ordied we ances idart,
Mall a my my toor spe sto of im whisertthind touc ands sweere cres a a
To orceethids ily mooven, sing tint of me herome int bimede here,
As toun, to ton asticge sto my ha

=== top-p = 0.9 (nucleus) ===
 
thy, mome ther selt ild canderave youd woulste trecoures,
O for beere hath thios orell soudsericest.

Nor por it Myemacces idy thomall nem thitsed we wost to a mad iseritht,
Whe con the weerut nous ould epradith and loom,
hond sily eigs of levellome int be cee here,
As toun, to thonest gor hat my ha


In [15]:
import torch, torch.nn as nn, torch.nn.functional as F
torch.manual_seed(1337)

text = open("shakespeare.txt").read()
chars = sorted(set(text)); vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}; itos = {i: c for c, i in stoi.items()}
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: "".join(itos[i] for i in ids)
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data)); train_data, val_data = data[:n], data[n:]

# --- model config (small, so it trains on a CPU in a couple of minutes) ---
block_size = 32     # context length
n_embd     = 64     # embedding size
n_heads    = 4      # attention heads
n_layers   = 3      # transformer blocks
batch_size = 16

class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
    def forward(self, x):
        B, T, C = x.shape
        k, q, v = self.key(x), self.query(x), self.value(x)
        wei = q @ k.transpose(-2, -1) * k.shape[-1] ** -0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wei = F.softmax(wei, dim=-1)
        return wei @ v

class MultiHeadAttention(nn.Module):
    def __init__(self, n_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(n_heads)])
        self.proj  = nn.Linear(n_heads * head_size, n_embd)
    def forward(self, x):
        return self.proj(torch.cat([h(x) for h in self.heads], dim=-1))

class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(n_embd, 4 * n_embd), nn.ReLU(), nn.Linear(4 * n_embd, n_embd))
    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd, n_heads):
        super().__init__()
        self.sa  = MultiHeadAttention(n_heads, n_embd // n_heads)
        self.ff  = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd); self.ln2 = nn.LayerNorm(n_embd)
    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

class GPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding    = nn.Embedding(vocab_size, n_embd)
        self.position_embedding = nn.Embedding(block_size, n_embd)
        self.blocks  = nn.Sequential(*[Block(n_embd, n_heads) for _ in range(n_layers)])
        self.ln_f    = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)
    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.token_embedding(idx) + self.position_embedding(torch.arange(T))
        x = self.ln_f(self.blocks(x))
        logits = self.lm_head(x)                       # (B, T, vocab)
        if targets is None:
            return logits, None
        loss = F.cross_entropy(logits.view(B * T, -1), targets.view(B * T))
        return logits, loss
    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -block_size:])
            probs = F.softmax(logits[:, -1, :], dim=-1)
            idx = torch.cat([idx, torch.multinomial(probs, 1)], dim=1)
        return idx

class MiniGPT:
    "Load a trained GPT and generate text from a prompt."
    def __init__(self, weights="gpt.pt"):
        self.model = GPT(); self.model.load_state_dict(torch.load(weights)); self.model.eval()

    @torch.no_grad()
    def complete(self, prompt="\n", max_new_tokens=300, temperature=0.8, top_k=20):
        idx = torch.tensor([encode(prompt)], dtype=torch.long)
        for _ in range(max_new_tokens):
            logits, _ = self.model(idx[:, -block_size:])
            logits = logits[:, -1, :] / temperature
            kth = torch.topk(logits, top_k)[0][:, [-1]]
            logits = logits.masked_fill(logits < kth, float("-inf"))
            probs = F.softmax(logits, dim=-1)
            idx = torch.cat([idx, torch.multinomial(probs, 1)], dim=1)
        return decode(idx[0].tolist())

gpt = MiniGPT()
print(gpt.complete(prompt="ROMEO:", max_new_tokens=200))

ROMEO:
Enceavere hond and, the muste nould lepon,
My nove ald mongor te dove she ill,
Of ghe weat is my lighs geere tis a thertit gourst,
heve his cene toud you of of gon hist oft hat fouls here,
Wher are'd
